1. 什么是自注意力，什么是交叉注意力？ 从K，Q，V的计算上和我说明白
2. 什么是位置编码，绝对位置编码和相对位置编码的区别是什么，了解下ROPE这个相对位置编码
3. 了解下transformer的源代码，看看具体的实现
4. transformer中有两类mask机制，去了解下什么是attention mask，什么是padding mask

## 自注意力和交叉注意力

在 10.1 中讲到，注意力机制将查询和键统一喂给 **注意力汇聚（attention pooling）** ，这些查询和键进行匹配，将希望得到的键值挑选出来。

在 10.2 中讲到，注意力汇聚的公式为：

$$
f(x) = \sum_{i=1}^n \alpha(x, x_i) y_i
$$



在 10.5 中讲到， **自注意力** 的核心是 **同一序列内部的元素间信息交互** ，即 **查询、键、值** 都来自同一个序列。

类比上面的注意力汇聚的公式，自注意力的输出为：

$$
\mathbf{y}_i = f\left(\mathbf{x}_i, (\mathbf{x}_1, \mathbf{x}_1), \dots, (\mathbf{x}_n, \mathbf{x}_n)\right) \in \mathbb{R}^d 
$$

$\mathbf{k}$ 、 $\mathbf{q}$ 、 $\mathbf{v}$ 的计算上都是同一输入使用不同权重矩阵 $\mathbf{W}_q$ 、 $\mathbf{W}_k$ 、 $\mathbf{W}_v$ 投影到不同空间得到。

而交叉注意力就是 10 章中一开始导入的注意力的例子，查询是一个序列，键值则来自另一个序列。Transformer 中的编码器-解码器注意力层即为此。

#### 代码讲解



下面使用教材中的代码进行样例讲解：

教材中使用多头注意力的类实现自注意力，即为多头自注意力。

```python
class MultiHeadAttention(nn.Module):
    """多头注意力"""
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 num_heads, dropout, bias=False, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.attention = d2l.DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        queries = transpose_qkv(self.W_q(queries), self.num_heads)
        keys = transpose_qkv(self.W_k(keys), self.num_heads)
        values = transpose_qkv(self.W_v(values), self.num_heads)
        output = self.attention(queries, keys, values, valid_lens)
        ...
```

```python
class EncoderBlock(nn.Module):
    """Transformer编码器块"""
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
                dropout, use_bias=False, **kwargs):
        ...
        self.attention = MultiHeadAttention(
                        key_size, query_size, value_size, num_hiddens, num_heads, dropout,
                        use_bias)
        ...

    def forward(self, X, valid_lens):
        Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
        return self.addnorm2(Y, self.ffn(Y))
```

可以看到，在 `EncoderBlock` 的前向传播中，多头自注意力层接收的三次 `X` 输入分别对应 $\mathbf{q}$ 、 $\mathbf{k}$ 、 $\mathbf{v}$ ，而这个前向传播实际上调用的是 `MultiHeadAttention` 的前向传播。

那么，后者使用到的 `W_q` `W_k` `W_v` 本来是实现加性注意力的（见 10.2），现在就可以看做转换为 $\mathbf{q}$ 、 $\mathbf{k}$ 、 $\mathbf{v}$ 的变换矩阵。

## 位置编码

位置编码在 10.5 中与自注意力一同讲到。在Transformer中，由于自注意力机制本身不考虑序列的顺序，因此需要注入位置信息。

- **绝对位置编码** ：为序列中每个绝对位置分配独特的编码向量
- **相对位置编码** ：编码元素间相对距离而非绝对位置

前者如 10 章中的正弦余弦位置编码。而后者因为要考虑 $i$ 位置和 $j$ 位置间的相对距离，这更适合在自注意力层中直接加入：因为 **查询×键 自然构建了形状近似为 $N \times N$ 的矩阵，可以直接与包含相对位置信息的矩阵相加** ：

$$
Attention = Softmax(QK^T + B)V
$$

$$
B_{i,j} = f(i-j)
$$

#### ROPE

在二维空间中，旋转矩阵定义为：（$\theta$ 为预设参数）

$$
R_m = 
\begin{bmatrix}
    \cos (m\theta) & -\sin (m\theta) \\
    \sin (m\theta) & \cos (m\theta) \\
\end{bmatrix}
$$

旋转操作为：

$$
RoPE(\mathbf{x}, m) = R_m \times \mathbf{x}
$$

现在考虑两个位置 $m$ 和 $n$ 处的向量 $q$ 和 $k$ ，它们根据自己的位置各有各的旋转矩阵，旋转后的点积为：

$$
<RoPE(\mathbf{q}_m, m), RoPE(\mathbf{k}_n, n)> = \\

(R_m \mathbf{q}_m)^T (R_n \mathbf{k}_n) = \mathbf{q}_m^T R_m^T R_n \mathbf{k}_n = \mathbf{q}_m^T R_{n-m} \mathbf{k}_n
$$

即点积结果还将依赖于相对位置 $n-m$ 对应的旋转矩阵。

对于d维向量，我们将向量分成d/2组二维分量，每组分别进行旋转。

在自注意力机制中，我们分别对查询向量q和键向量k应用旋转位置编码，然后计算注意力分数 $<RoPE(\mathbf{q}_m, m), RoPE(\mathbf{k}_n, n)>$ 。

ROPE不添加额外的位置向量，而是直接修改词向量本身。

## mask机制

#### padding mask填充掩码

在 10.2 中介绍到掩蔽softmax操作：

在批量处理中，不同序列长度不同，通常用填充（padding）使长度一致，掩蔽 softmax 可以忽略这些填充位置。

给定一个输入向量（或矩阵） $x$ 和一个掩码 $mask$ （与x形状相同），对需要掩蔽的位置，将 $x$ 的对应元素设置为一个极小的值（如 $-1e9$ ），这样在计算 softmax 时，这些位置的指数值接近于 $0$ ，然后计算标准的 softmax 函数。

#### attention mask注意力掩码

在 10.6 中介绍到，自注意力中每个位置只能考虑该位置之前的所有位置，即为注意力掩码（这种为了自回归的掩码也称因果掩码）。

## Transformer简单实现